# Phase 3B Extension — Oxidation State and Open Metal Site Investigation

This notebook investigates whether oxidation-state and possible open-metal-site descriptors can be assigned reliably from CIF structures.

These descriptors are not added automatically to the final modeling dataset unless the assignment method is reproducible, interpretable, and reliable.

In [1]:
# Import libraries needed for data loading and CIF structure parsing.
import os
from pathlib import Path
import pandas as pd
from pymatgen.core import Structure
from tqdm import tqdm

In [2]:
# Load the latest Phase 3 dataset.
# This dataset contains CIF paths, metal descriptors, and structure-derived descriptors.

phase3_path = "/home/susan/mof-co2-adsorption/data/processed/hmof_phase3_structure_descriptors.csv"

df = pd.read_csv(phase3_path, low_memory=False)

print(df.shape)
print()
print('The latest Phase 3 dataset was loaded successfully with 25,928 MOFs and 75 ')
df.head()

(25928, 75)

The latest Phase 3 dataset was loaded successfully with 25,928 MOFs and 75 


,filename,lcd,pld,void_fraction,surface_area_m2g,mofid,CO2_uptake_0.01bar_molkg,CO2_uptake_0.05bar_molkg,CO2_uptake_0.1bar_molkg,CO2_uptake_0.5bar_molkg,...,avg_metal_atomic_mass,unit_cell_volume,density,num_sites,lattice_a,lattice_b,lattice_c,lattice_alpha,lattice_beta,lattice_gamma
0,hMOF-0,11.75,10.75,0.795539,3676.3,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,0.022406,0.073552,0.225851,0.885221,...,65.409,2077.257842,0.615515,53,12.759401,12.759399,12.759393,89.967969,89.980163,90.016641
1,hMOF-1,4.75,3.25,0.388986,580.7,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,0.173407,0.925813,1.806490,4.779010,...,65.409,2077.257842,1.231029,106,12.759401,12.759399,12.759393,89.967969,89.980163,90.016641
2,hMOF-10,4.75,3.75,0.307505,377.6,[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Z...,0.651164,2.129830,2.817220,3.909470,...,65.409,2077.258057,1.489894,106,12.759401,12.759400,12.759393,90.030930,90.018575,90.015071
3,hMOF-100,9.25,8.25,0.731919,3399.2,COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1...,0.031321,0.143123,0.332034,1.756380,...,65.409,2077.258072,0.711524,69,12.759401,12.759400,12.759393,89.969639,89.980663,90.013730
4,hMOF-1000,3.75,2.75,0.323803,85.2,CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=...,0.144024,0.698367,1.402230,2.694200,...,65.409,1591.808592,1.564747,110,12.759401,11.169416,11.169409,90.030874,90.021365,90.011865


In [3]:
# Check that the dataset contains the key columns needed for oxidation-state and OMS investigation.
# We need filename, mofid, cif_path, and metal information.

required_columns = [
    "filename",
    "mofid",
    "cif_path",
    "metal_elements",
    "num_metal_types",
    "total_metal_atoms",
]

for column in required_columns:
    print(column, "->", column in df.columns)
print(" All required columns for oxidation-state and OMS investigation are present in the dataset.")

filename -> True
mofid -> True
cif_path -> True
metal_elements -> True
num_metal_types -> True
total_metal_atoms -> True
 All required columns for oxidation-state and OMS investigation are present in the dataset.


In [4]:
# Select one sample MOF for oxidation-state testing.
# We start with one CIF file before applying any method to multiple structures.

sample_index = 0

sample_filename = df.loc[sample_index, "filename"]
sample_cif_path = df.loc[sample_index, "cif_path"]
sample_metal_elements = df.loc[sample_index, "metal_elements"]

print("Filename:", sample_filename)
print("CIF path:", sample_cif_path)
print("Metal elements:", sample_metal_elements)

Filename: hMOF-0
CIF path: /home/susan/mof-co2-adsorption/data/raw/hmof_json_local/hMOF-0.cif
Metal elements: ['Zn']


In [5]:
# Load the selected sample CIF structure with pymatgen.
# This structure will be used to test oxidation-state assignment.

sample_structure = Structure.from_file(sample_cif_path)

print(sample_structure.composition)
# Try assigning oxidation states using pymatgen's built-in oxidation-state guessing.
# We use a copy so the original structure remains unchanged.

sample_structure_oxi = sample_structure.copy()

try:
    sample_structure_oxi.add_oxidation_state_by_guess()
    print(sample_structure_oxi.composition)
    print("Oxidation-state assignment succeeded.")
except Exception as error:
    print("Oxidation-state assignment failed.")
    print(error)

# Extract oxidation states only for metal sites.
# We focus on metal centers because they are most relevant to CO2 binding environments.

metal_oxidation_states = []

for site in sample_structure_oxi:
    species = site.specie
    
    if species.element.is_metal:
        metal_oxidation_states.append(species.oxi_state)

print(metal_oxidation_states)

Zn4 H12 C24 O13
Zn2+4 H+12 C0.25+24 O2-13
Oxidation-state assignment succeeded.
[2.0, 2.0, 2.0, 2.0]


In [6]:
# Summarize the sample metal oxidation states.
# These summary values are easier to use as machine-learning descriptors.

sample_mean_metal_oxidation_state = sum(metal_oxidation_states) / len(metal_oxidation_states)
sample_min_metal_oxidation_state = min(metal_oxidation_states)
sample_max_metal_oxidation_state = max(metal_oxidation_states)

print("Mean metal oxidation state:", sample_mean_metal_oxidation_state)
print("Min metal oxidation state:", sample_min_metal_oxidation_state)
print("Max metal oxidation state:", sample_max_metal_oxidation_state)

Mean metal oxidation state: 2.0
Min metal oxidation state: 2.0
Max metal oxidation state: 2.0


In [21]:
import re
from pymatgen.core import Composition, Element, Structure

def extract_metal_oxidation_descriptors(cif_path):
    """
    Ultra-fast heuristic extraction of metal oxidation states from a CIF file.
    Parses chemical formula directly from file headers to avoid slow 3D parsing.
    """
    try:
        formula_str = None
        
        # Read CIF text line-by-line to extract the formula header directly
        with open(cif_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if line.startswith("_chemical_formula_sum") or line.startswith("_chemical_formula_structural"):
                    match = re.search(r"['\"](.*?)['\"]", line)
                    if match:
                        formula_str = match.group(1)
                    else:
                        parts = line.strip().split(maxsplit=1)
                        if len(parts) > 1:
                            formula_str = parts[1]
                    break
        
        # Fallback to structure parsing only if header formula is missing
        if not formula_str:
            comp = Structure.from_file(cif_path).composition
        else:
            clean_formula = re.sub(r"[_\{\}]", "", formula_str)
            comp = Composition(clean_formula)

        metal_oxidation_states = []
        
        for element, count in comp.items():
            el = Element(str(element))
            if el.is_metal:
                common_states = el.common_oxidation_states
                if common_states:
                    pos_states = [s for s in common_states if s > 0]
                    assigned_state = pos_states[0] if pos_states else common_states[0]
                else:
                    assigned_state = 2  # Standard fallback valence
                
                metal_oxidation_states.extend([assigned_state] * int(round(count)))
        
        if metal_oxidation_states:
            mean_val = sum(metal_oxidation_states) / len(metal_oxidation_states)
            min_val = min(metal_oxidation_states)
            max_val = max(metal_oxidation_states)
        else:
            mean_val, min_val, max_val = None, None, None
            
        return {
            "oxidation_assignment_success": True if metal_oxidation_states else False,
            "mean_metal_oxidation_state": mean_val,
            "min_metal_oxidation_state": min_val,
            "max_metal_oxidation_state": max_val,
            "oxidation_error": None
        }
        
    except Exception as error:
        return {
            "oxidation_assignment_success": False,
            "mean_metal_oxidation_state": None,
            "min_metal_oxidation_state": None,
            "max_metal_oxidation_state": None,
            "oxidation_error": str(error)
        }

In [22]:
# Test the oxidation-state descriptor function on the sample CIF.
# This confirms that the function returns the expected Zn(II) oxidation-state descriptors.

sample_oxidation_descriptors = extract_metal_oxidation_descriptors(sample_cif_path)

print(sample_oxidation_descriptors)

{'oxidation_assignment_success': True, 'mean_metal_oxidation_state': 2.0, 'min_metal_oxidation_state': 2, 'max_metal_oxidation_state': 2, 'oxidation_error': None}


In [23]:
# Test the oxidation-state descriptor function on the first 5 CIF files.
# This checks whether the function works across multiple MOFs before processing the full dataset.

test_oxidation_results = []

for cif_path in df["cif_path"].head(5):
    descriptors = extract_metal_oxidation_descriptors(cif_path)
    test_oxidation_results.append(descriptors)

pd.DataFrame(test_oxidation_results)


,oxidation_assignment_success,mean_metal_oxidation_state,min_metal_oxidation_state,max_metal_oxidation_state,oxidation_error
0,True,2.0,2,2,None
1,True,2.0,2,2,None
2,True,2.0,2,2,None
3,True,2.0,2,2,None
4,True,2.0,2,2,None


In [24]:
# Apply the oxidation-state descriptor function to all CIF files.
# This records both successful and failed oxidation-state assignments.

oxidation_results = []

for cif_path in df["cif_path"]:
    descriptors = extract_metal_oxidation_descriptors(cif_path)
    oxidation_results.append(descriptors)

print("Number of oxidation descriptor rows:", len(oxidation_results))

/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CI

Number of oxidation descriptor rows: 25928


In [25]:
# Convert oxidation-state results into a DataFrame.
# Each row corresponds to one MOF in the same order as df.

oxidation_df = pd.DataFrame(oxidation_results)

print(oxidation_df.shape)
oxidation_df.head()

(25928, 5)


,oxidation_assignment_success,mean_metal_oxidation_state,min_metal_oxidation_state,max_metal_oxidation_state,oxidation_error
0,True,2.0,2,2,None
1,True,2.0,2,2,None
2,True,2.0,2,2,None
3,True,2.0,2,2,None
4,True,2.0,2,2,None


In [26]:
# Add oxidation-state descriptor columns directly to the existing df.
# This keeps the Phase 3 dataset in one dataframe.

oxidation_cols = oxidation_df.columns.tolist()

df[oxidation_cols] = oxidation_df[oxidation_cols]

print(df.shape)
df[["filename"] + oxidation_cols].head()

(25928, 80)


,filename,oxidation_assignment_success,mean_metal_oxidation_state,min_metal_oxidation_state,max_metal_oxidation_state,oxidation_error
0,hMOF-0,True,2.0,2,2,None
1,hMOF-1,True,2.0,2,2,None
2,hMOF-10,True,2.0,2,2,None
3,hMOF-100,True,2.0,2,2,None
4,hMOF-1000,True,2.0,2,2,None


In [28]:
# Save the Phase 3 dataset with oxidation-state investigation descriptors.
# This file can be used to test whether oxidation-state descriptors improve Model B.

phase3_structure_path = "/home/susan/mof-co2-adsorption/data/processed/hmof_phase3_structure_descriptors.csv"


df.to_csv(phase3_structure_path, index=False)

print("Saved updated file:", phase3_structure_path)
print("Dataset shape:", df.shape)

Saved updated file: /home/susan/mof-co2-adsorption/data/processed/hmof_phase3_structure_descriptors.csv
Dataset shape: (25928, 80)


In [29]:
# Check the added oxidation-state heuristic columns.

oxidation_cols = [
    "oxidation_assignment_success",
    "mean_metal_oxidation_state",
    "min_metal_oxidation_state",
    "max_metal_oxidation_state",
    "oxidation_error",
]

df[["filename"] + oxidation_cols].head()

,filename,oxidation_assignment_success,mean_metal_oxidation_state,min_metal_oxidation_state,max_metal_oxidation_state,oxidation_error
0,hMOF-0,True,2.0,2,2,None
1,hMOF-1,True,2.0,2,2,None
2,hMOF-10,True,2.0,2,2,None
3,hMOF-100,True,2.0,2,2,None
4,hMOF-1000,True,2.0,2,2,None


In [30]:
# Check the success rate and distribution of estimated metal oxidation states.
# This helps confirm whether the heuristic produced values consistently across the full dataset.

print(df["oxidation_assignment_success"].value_counts(dropna=False))
print()
print(df["mean_metal_oxidation_state"].value_counts(dropna=False).head(20))

oxidation_assignment_success
True    25928
Name: count, dtype: int64

mean_metal_oxidation_state
2.0    25928
Name: count, dtype: int64


Conclusion: The fast heuristic oxidation-state method assigned +2 to all MOFs, producing no feature variation. Therefore, oxidation-state descriptors will not be included in Model B because they cannot improve prediction in their current form.